
# Authoring a new physics block with ``SEDModelComponent``

The on-ramp for adding a custom physics block to tengri. Subclass
:class:`SEDModelComponent`, declare ``name``, ``parameter_prefix``,
priors as class attributes, and implement ``predict(p, sed_in, wave)``.
``__init_subclass__`` registers the new variant and auto-fills the
``inputs()`` / ``outputs()`` contracts.

To show the workflow end-to-end without dragging in the full
``SEDModel.build`` plumbing, we write a *modified Calzetti* law with an
explicit 2175 Å UV bump (Noll+2009 Eq. 4), invoke ``predict`` directly
on a flat input spectrum, and plot the resulting attenuation curve
alongside the bare Calzetti and the Cardelli+1989 MW shape. The bump
is parametrized by amplitude $E_b$ and FWHM $\gamma$;
setting $E_b=0$ recovers Calzetti exactly.


In [ ]:
import warnings

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from tengri import SEDModelComponent, Uniform
from tengri.dust import calzetti as _calzetti_law, cardelli as _cardelli_law
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")


class CalzettiPlusBump(SEDModelComponent):
    r"""Calzetti law plus a Drude-profile UV bump at 2175 Å.

    The attenuation factor is :math:`e^{-\tau_V k_{\rm eff}(\lambda)}` with

    .. math::

       k_{\rm eff}(\lambda) = k_{\rm Calz}(\lambda)
           + \frac{E_b\,\gamma^2 \lambda^2}
                   {(\lambda^2 - \lambda_0^2)^2 + \gamma^2 \lambda^2},

    :math:`\lambda_0 = 2175` Å. :math:`E_b = 0` recovers bare Calzetti.
    """

    name = "calzetti_bump"
    parameter_prefix = "dust_"

    tau_v = Uniform(0.0, 4.0, description="V-band optical depth", units="dimensionless")
    eb = Uniform(0.0, 5.0, description="2175 A bump amplitude", units="dimensionless")
    gamma = Uniform(100.0, 600.0, description="bump FWHM", units="Angstrom")

    inputs = {}  # noqa: RUF012
    outputs = {"L_absorbed": "erg/s"}  # noqa: RUF012

    def predict(self, p, sed_in, wave):
        k_calz = _calzetti_law(wave)
        lam2 = wave**2
        lam0_sq = 2175.0**2
        drude = p["eb"] * p["gamma"] ** 2 * lam2 / ((lam2 - lam0_sq) ** 2 + p["gamma"] ** 2 * lam2)
        k_eff = k_calz + drude
        atten = jnp.exp(-p["tau_v"] * k_eff)
        sed_out = sed_in * atten
        c = 2.99792458e18
        nu = c / wave
        L_absorbed = jnp.trapezoid((sed_in - sed_out)[::-1], nu[::-1])
        return sed_out, {"L_absorbed": L_absorbed}


# ─── Apply each law to a flat input SED on a common wavelength grid ──────────
wave = jnp.linspace(1.0e3, 9.0e3, 1000)
flat_sed = jnp.ones_like(wave)
TAU_V = 0.6

# Built-in Calzetti
k_calz = np.asarray(_calzetti_law(wave))
A_calz = np.exp(-TAU_V * k_calz)

# Built-in Cardelli MW
k_card = np.asarray(_cardelli_law(wave))
A_card = np.exp(-TAU_V * k_card)

# Custom subclass — call predict directly
mybump = CalzettiPlusBump()
p = {"tau_v": TAU_V, "eb": 2.5, "gamma": 300.0}
sed_out, published = mybump.predict(p, flat_sed, wave)
A_bump = np.asarray(sed_out)

# Sanity print so the gallery card body shows the registration.
print(
    f"SEDModelComponent registered as '{CalzettiPlusBump.name}' with parameters {sorted(p.keys())}"
)
print(f"  L_absorbed published in state.derived: {float(published['L_absorbed']):.3e} erg/s")

# ─── Figure ──────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7.0, 4.4))
wave_np = np.asarray(wave)
ax.plot(wave_np, A_calz, color="C0", lw=1.5, label="Calzetti+2000 (built-in)")
ax.plot(wave_np, A_card, color="C2", lw=1.5, label="Cardelli+1989 MW (built-in)")
ax.plot(wave_np, A_bump, color="C3", lw=1.5, label=r"Calzetti + 2175 Å bump  (custom, $E_b=2.5$)")
ax.axvline(2175.0, color="0.6", lw=0.5, ls="--")
ax.text(2175.0, 0.9, " 2175 Å", color="0.4", fontsize=8, va="top")
ax.set_xlabel(r"Rest-frame wavelength  [$\mathrm{\AA}$]")
ax.set_ylabel(r"Attenuation  $A(\lambda) = e^{-\tau_V k_{\rm eff}(\lambda)}$")
ax.set_ylim(0.0, 1.05)
ax.legend(frameon=False, fontsize=9, loc="lower right")

fig.savefig("plot_custom_attenuation_component.png", dpi=150, bbox_inches="tight")